## Dense baseline — end-to-end stage

Reads the candidate file written by `baseline/retrieval.ipynb` (n candidates
per mention, retrieved from mention + context with no feedback), asks a local
Qwen model to **rank those n candidates** using the mention and its context,
and reports:

- `R@1`, `R@5`: candidate retrieval before the selector,
- `acc@1`: end-to-end accuracy after the LLM re-ranking (the selector's first
  CUI equals the gold CUI),
- `acc@5`: share of mentions whose gold CUI is among the selector's first 5,

overall and per label.

Nothing from `source/GRF/` is imported: only `source/common/` and
`source/baseline/selector.py`.

In [ ]:
%pip install "transformers>=4.44" "accelerate>=0.34" "tqdm>=4.66"

## Import Libraries

Third-party imports only. The shared `common` package lives inside the
repository clone, so project modules are imported at the end of the
repository-setup stage below.


In [ ]:
from __future__ import annotations

import gc
import json
import os
import subprocess
import sys
import time
from contextlib import contextmanager
from dataclasses import asdict
from pathlib import Path

from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("tqdm:", __import__("tqdm").__version__)


@contextmanager
def progress_bar(total, desc: str, unit: str = "item"):
    """Drive a tqdm bar from a progress_callback(done, total)."""
    bar = tqdm(total=total, desc=desc, unit=unit)

    def report(done: int, total=None) -> None:
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        bar.close()

## Constants and Configuration

This stage reads and writes in one folder — `COMMON_DIR/output/{METHOD}/`
with `METHOD = "baseline"` — because its input is the retrieval output:

- `CANDIDATES_PATH`: the JSONL written by `baseline/retrieval.ipynb`,
- `RESULTS_OUTPUT_PATH`: the metrics JSON written by this notebook,
- `SELECTOR_MODEL`: Qwen checkpoint (Hugging Face repo id or local directory)
  loaded with transformers, bf16, greedy decoding, no quantization,
- `SELECTOR_BATCH_SIZE`: prompts per `generate` call,
- `MAX_SAMPLES`: `None` = every mention, an int truncates for a smoke run.

In [ ]:
# ---------------------------------------------------------------------------
# Git repository
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive layout: input and output share COMMON_DIR/output/{METHOD}/
# ---------------------------------------------------------------------------
METHOD = "baseline"
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
COMMON_DIR = DRIVE_ROOT / "work" / "nlp" / "UMLS" / "entity_linking"
OUTPUT_DIR = COMMON_DIR / "output" / METHOD

CANDIDATES_PATH = OUTPUT_DIR / "baseline_candidates.jsonl"      # from retrieval.ipynb
RESULTS_OUTPUT_PATH = OUTPUT_DIR / "baseline_e2e_results.json"

# ---------------------------------------------------------------------------
# Selector LLM (local Qwen, transformers only, no quantization)
# ---------------------------------------------------------------------------
SELECTOR_MODEL = "Qwen/Qwen3.8-27B"        # repo id HF hoặc đường dẫn local
DTYPE = "bfloat16"                      # A100 native; float16 | float32 optional
MAX_NEW_TOKENS = 512                    # mỗi câu chỉ cần trả JSON ranking
SELECTOR_BATCH_SIZE = 8                 # prompts per generate call
MAX_SAMPLES = None                      # None = all mentions; int = smoke run

# ---------------------------------------------------------------------------
# Runtime
# ---------------------------------------------------------------------------
CACHE_DIR = Path("/content/baseline_cache")
HF_HOME = CACHE_DIR / "huggingface"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))

_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value


log("Configuration loaded.")
print(f"Repository   : {GIT_REPO_URL} ({GIT_BRANCH})")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Candidates   : {CANDIDATES_PATH}")
print(f"Results out  : {RESULTS_OUTPUT_PATH}")
print(f"Selector     : {SELECTOR_MODEL} ({DTYPE}, max_new_tokens={MAX_NEW_TOKENS})")

## Stage 1 — Clone or Update the Code Repository

Colab starts from an empty runtime, so the notebook clones the code repository
and imports the shared modules from `source/common/`. Re-running this cell
updates the existing clone instead of downloading it again.


In [ ]:
repo_url = str(require_configured(GIT_REPO_URL, "GIT_REPO_URL"))


def _git(*args: str) -> None:
    log("git " + " ".join(args))
    try:
        subprocess.run(["git", *args], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"git {args[0]} failed with exit code {exc.returncode}.") from exc


if REPO_CLONE_DIR.exists() and not (REPO_CLONE_DIR / ".git").exists():
    raise RuntimeError(
        f"{REPO_CLONE_DIR} exists but is not a git clone. "
        "Remove it or point REPO_CLONE_DIR elsewhere."
    )

if (REPO_CLONE_DIR / ".git").exists():
    log(f"Repository already present at {REPO_CLONE_DIR}; updating branch {GIT_BRANCH!r}.")
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    log(f"Cloning {repo_url} (branch {GIT_BRANCH!r}) into {REPO_CLONE_DIR} ...")
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, repo_url, str(REPO_CLONE_DIR))

SOURCE_DIR = REPO_CLONE_DIR / "source"
if not (SOURCE_DIR / "common" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"{SOURCE_DIR / 'common' / '__init__.py'} is missing; check GIT_BRANCH and the repository layout."
    )
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from baseline.selector import SelectionError, build_prompt, parse_ranking
from common.candidates import load_candidates
from common.evaluator import accuracy_at_k, metrics_from_responses
from common.local_llm import LocalCausalLM
from common.schemas import Response

log(f"Repository ready at {REPO_CLONE_DIR}; common and baseline imported from {SOURCE_DIR}.")

## Stage 2 — Mount Google Drive and Locate the Candidate File

The candidate file and the results output live on Drive, not in the repository.
This stage mounts Drive and checks the input plus the output path before any
record is parsed.

In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

candidates_path = Path(require_configured(CANDIDATES_PATH, "CANDIDATES_PATH"))
results_path = Path(require_configured(RESULTS_OUTPUT_PATH, "RESULTS_OUTPUT_PATH"))

if not candidates_path.is_file():
    raise FileNotFoundError(
        f"Candidate file not found: {candidates_path}. Run baseline/retrieval.ipynb first."
    )
if candidates_path.suffix.lower() != ".jsonl":
    raise ValueError(f"CANDIDATES_PATH must end in .jsonl, got {candidates_path.name!r}.")
if results_path.suffix.lower() != ".json":
    raise ValueError(f"RESULTS_OUTPUT_PATH must end in .json, got {results_path.name!r}.")
log(f"Candidates: {candidates_path} ({candidates_path.stat().st_size / 1024 ** 2:.2f} MB)")
log(f"Results out: {results_path}")

## Stage 3 — Load Saved Candidates

`load_candidates` restores the gold samples and the retrieval responses from the
single JSONL file. Nothing is re-encoded or re-retrieved. It reports each record
through `progress_callback`, which drives the bar.

In [ ]:
log(f"Loading candidates from {candidates_path} ...")
started = time.time()
with progress_bar(None, "Loading candidate records", "record") as report:
    samples, responses = load_candidates(candidates_path, progress_callback=report)
log(
    f"Loaded {len(samples):,} samples and {len(responses):,} responses "
    f"in {time.time() - started:.1f}s."
)
if not samples:
    raise ValueError(f"{candidates_path} contains no candidate records.")
if MAX_SAMPLES is not None:
    if isinstance(MAX_SAMPLES, bool) or not isinstance(MAX_SAMPLES, int) or MAX_SAMPLES < 1:
        raise ValueError("MAX_SAMPLES must be None or a positive integer.")
    samples = samples[:MAX_SAMPLES]
    responses = responses[:MAX_SAMPLES]
empty = sum(1 for response in responses if not response.candidates)
log(f"Mentions with candidates: {len(responses) - empty:,}; without: {empty:,}.")
if empty:
    log(f"WARNING: {empty:,} mention(s) have no candidates and keep an empty ranking.")

## Stage 4 — Load the Selector Model

`LocalCausalLM` loads `SELECTOR_MODEL` with transformers: bf16 weights,
`device_map="auto"`, greedy decoding and the chat template with thinking
disabled. The model only ranks the n saved candidates — no encoder, index or
KB artifact is loaded here.

In [ ]:
log(f"Loading selector {SELECTOR_MODEL!r} on {DTYPE} ...")
started = time.time()
llm = LocalCausalLM(SELECTOR_MODEL, max_new_tokens=MAX_NEW_TOKENS, dtype=DTYPE)
log(
    f"Selector ready on {llm.device} in {time.time() - started:.1f}s; "
    f"batch={SELECTOR_BATCH_SIZE}."
)

## Stage 5 — LLM Selection over the n Candidates

For each mention the selector gets the mention, its context and the n candidates
saved by the retrieval stage, and must return a full ranking of those CUIs (JSON
`{"ranking": [...]}`). `parse_ranking` maps the answer back onto the candidate
list; anything the model omits is appended in retrieval order, so the result is
always a complete ranking.

The selector's first CUI is the end-to-end answer; Stage 6 reports how often it
differs from the retrieval top-1. If the answer contains no candidate CUI at
all, the mention falls back to the retrieval order and is counted in
`selection_failures` — it is never dropped.

In [ ]:
def select_rankings(samples, responses, report):
    """Ask the LLM to rank the n candidates; fall back to retrieval order."""
    reranked = []
    failures = 0
    top1_changed = 0
    for start in range(0, len(samples), SELECTOR_BATCH_SIZE):
        batch = list(zip(
            samples[start : start + SELECTOR_BATCH_SIZE],
            responses[start : start + SELECTOR_BATCH_SIZE],
        ))
        prompts = [
            build_prompt(sample.mention, sample.context, response.candidates)
            for sample, response in batch
        ]
        answers = llm.complete_batch(prompts)

        for (sample, response), answer in zip(batch, answers):
            cuis = [candidate.cui for candidate in response.candidates]
            if not cuis:
                reranked.append(Response(cui="", candidates=(), debug={"concept_ranking": []}))
                failures += 1
                continue

            try:
                ranking = parse_ranking(answer, cuis)
            except SelectionError as exc:
                failures += 1
                ranking = cuis
                if failures <= 3:
                    log(
                        f"WARNING: record {sample.record_index} entity "
                        f"{sample.entity_index}: {exc}"
                    )

            by_cui = {candidate.cui: candidate for candidate in response.candidates}
            ordered = tuple(by_cui[cui] for cui in ranking)
            if ordered[0].cui != cuis[0]:
                top1_changed += 1
            reranked.append(
                Response(
                    cui=ordered[0].cui,
                    candidates=ordered,
                    debug={
                        "concept_ranking": [
                            {
                                "cui": candidate.cui,
                                "score": candidate.score,
                                "surface": candidate.surface,
                            }
                            for candidate in ordered
                        ],
                    },
                )
            )
        report(min(start + len(batch), len(samples)), len(samples))
    return reranked, failures, top1_changed


with progress_bar(len(samples), "LLM selection", "mention") as report:
    started = time.time()
    reranked, selection_failures, top1_changed = select_rankings(
        samples, responses, report
    )

log(
    f"Re-ranked {len(reranked):,} mentions in {time.time() - started:.1f}s: "
    f"final CUI changed for {top1_changed:,}, "
    f"{selection_failures:,} fallback(s)."
)

## Stage 6 — Retrieval and End-to-End Metrics

Two metric groups over the same samples:

- **R@1/R@5**: candidate retrieval before the selector, from the untruncated
  concept ranking written by `baseline/retrieval.ipynb`,
- **acc@1/acc@5**: the LLM re-ranking result — acc@1 is the share of mentions
  whose final CUI (the selector's first choice) equals the gold CUI
  (architecture §10, end-to-end accuracy), acc@5 the share whose gold CUI is
  among the first 5 concepts the selector ranked.

The printed fallback count tells how often the selector's answer could not be
parsed; those mentions keep the retrieval order, so their acc@k cannot differ
from R@k. Everything is reported overall and per label, then saved to
`RESULTS_OUTPUT_PATH`.

In [ ]:
retrieval_overall, retrieval_by_label = metrics_from_responses(samples, responses)
e2e_overall, e2e_by_label = metrics_from_responses(samples, reranked)
acc5_overall, acc5_by_label = accuracy_at_k(samples, reranked, 5)


def _row(label, retrieval, e2e, acc5) -> str:
    return (
        f"{str(label):<16}{retrieval.n:>7}"
        f"{100 * retrieval.recall_at_1:>9.2f}{100 * retrieval.recall_at_5:>9.2f}"
        f"{100 * e2e.accuracy:>9.2f}{100 * acc5:>9.2f}"
    )


print(f"{'label':<16}{'n':>7}{'R@1':>9}{'R@5':>9}{'acc@1':>9}{'acc@5':>9}")
print(_row("ALL", retrieval_overall, e2e_overall, acc5_overall))
for label, retrieval_metrics in retrieval_by_label.items():
    print(_row(label, retrieval_metrics, e2e_by_label[label], acc5_by_label[label]))

print(
    f"LLM re-rank: final CUI differs from the retrieval top-1 for "
    f"{top1_changed:,}/{len(samples):,} mentions; "
    f"{selection_failures:,} fallback(s) kept the retrieval order."
)

payload = {
    "candidates_file": str(candidates_path),
    "selector_model": SELECTOR_MODEL,
    "n_samples": len(samples),
    "selection_failures": selection_failures,
    "top1_changed": top1_changed,
    "retrieval": {
        "overall": asdict(retrieval_overall),
        "by_label": {str(k): asdict(v) for k, v in retrieval_by_label.items()},
    },
    "e2e": {
        "overall": asdict(e2e_overall),
        "by_label": {str(k): asdict(v) for k, v in e2e_by_label.items()},
        "acc5_overall": acc5_overall,
        "acc5_by_label": {str(k): v for k, v in acc5_by_label.items()},
    },
}

results_path.parent.mkdir(parents=True, exist_ok=True)
log(f"Saving results to {results_path} ...")
results_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
saved = json.loads(results_path.read_text(encoding="utf-8"))
assert saved["retrieval"]["overall"] == payload["retrieval"]["overall"]
assert saved["e2e"]["overall"] == payload["e2e"]["overall"]
assert saved["e2e"]["acc5_overall"] == payload["e2e"]["acc5_overall"]
log(f"Saved and verified {results_path} ({results_path.stat().st_size / 1024 ** 2:.2f} MB).")
log("Baseline end-to-end evaluation complete.")

## Stage 7 — Release Memory

The metrics are saved, so drop the parsed samples/responses, the re-ranked
responses and the selector model, then empty the CUDA cache.

In [ ]:
for name in (
    "llm",
    "samples",
    "responses",
    "reranked",
    "payload",
    "saved",
    "retrieval_overall",
    "retrieval_by_label",
    "e2e_overall",
    "e2e_by_label",
    "acc5_overall",
    "acc5_by_label",
    "top1_changed",
):
    globals().pop(name, None)
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_b, total_b = torch.cuda.mem_get_info()
        log(
            f"CUDA cache emptied: {free_b / 1024 ** 3:.2f}/"
            f"{total_b / 1024 ** 3:.2f} GiB free."
        )
except ImportError:
    pass

log("Baseline E2E memory released (selector, samples, responses).")